# ML Model Building - Customer Churn Prediction

This notebook builds and compares multiple classification models to predict customer churn.

**Steps covered:**
1. Baseline model (no cleaning)
2. Data cleaning and feature engineering
3. Feature scaling (StandardScaler, MinMaxScaler)
4. Handling imbalanced data (SMOTEENN, SMOTE, ADASYN)
5. Model comparison (DecisionTree, RandomForest, XGBoost, AdaBoost, CatBoost, LightGBM)
6. Hyperparameter tuning (RandomizedSearchCV, Optuna)
7. Save the best model as a .pkl file

## 1. Import Libraries

We import all libraries upfront for clarity.
sklearn provides the core ML tools.
imblearn handles class imbalance.
xgboost, lightgbm, catboost are gradient boosting frameworks.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import joblib

%matplotlib inline
print("All libraries imported successfully")

All libraries imported successfully


## 2. Load Data

Paths are resolved relative to this notebook's location so the project runs from any directory.

In [2]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_PATH = os.path.join(NOTEBOOK_DIR, 'data', 'Customer-Churn.csv')
MODELS_DIR = os.path.join(NOTEBOOK_DIR, 'models')

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head(5)

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Churn rate
df.Churn.value_counts() / len(df) * 100
# Expected: ~73% No, ~27% Yes -> 26.53% churn rate

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

## 3. Baseline Model (Before Cleaning)

We first build a quick baseline to understand the starting point.
This uses the raw data without fixing TotalCharges or creating tenure bins.

**Purpose:** Show that base accuracy looks good (~76%) but is unreliable
because the dataset is imbalanced - a model that always predicts 'No Churn'
would achieve 73% accuracy trivially.

In [4]:
# Define X (features) and y (target)
# Drop customerID (non-predictive ID) and Churn (target)
y_base = df['Churn']
X_base = df.drop(columns=['customerID', 'Churn'])

print("Shape of X:", X_base.shape)
print("Shape of y:", y_base.shape)

Shape of X: (7043, 19)
Shape of y: (7043,)


In [5]:
# One-hot encode categorical columns
# drop_first=True removes the first dummy column to avoid multicollinearity
X_base = pd.get_dummies(X_base, drop_first=True)

# Map target to numeric
y_base = df['Churn'].map({'No': 0, 'Yes': 1})

print("X shape after encoding:", X_base.shape)

X shape after encoding: (7043, 6559)


In [6]:
# Train-test split: 80% train, 20% test
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

# Decision Tree - no hyperparameter tuning, default settings
model_dt = DecisionTreeClassifier()
model_dt.fit(X_train_base, y_train_base)
y_pred_dt = model_dt.predict(X_test_base)

print("=== Baseline Decision Tree (no data cleaning) ===")
print(classification_report(y_test_base, y_pred_dt))

=== Baseline Decision Tree (no data cleaning) ===
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1036
           1       0.58      0.49      0.53       373

    accuracy                           0.77      1409
   macro avg       0.70      0.68      0.69      1409
weighted avg       0.76      0.77      0.76      1409



### Initial Insights from Baseline:

- Accuracy ~76-77% - looks acceptable but misleading
- Recall for class 1 (Churn) is low - the model misses many churners
- Root causes to address:
  1. TotalCharges needs to be numeric (data cleaning)
  2. Feature scaling is needed for distance-based and gradient methods
  3. Class imbalance needs to be handled (73:27 ratio)

In [7]:
df.info()
# Observe: TotalCharges is dtype object, should be float

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 4. Data Cleaning

Replicating the same cleaning steps as the EDA notebook to ensure consistency.

In [8]:
# Work on a copy to preserve raw df
telco_data = df.copy()

In [9]:
# Convert TotalCharges to numeric; whitespace entries become NaN
telco_data.TotalCharges = pd.to_numeric(telco_data.TotalCharges, errors='coerce')
telco_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [10]:
# Identify rows with NaN (11 rows, 0.15% of data - safe to drop)
telco_data.loc[telco_data['TotalCharges'].isnull() == True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [11]:
# Remove rows with any missing value
telco_data.dropna(how='any', inplace=True)
print("Shape after dropna:", telco_data.shape)

Shape after dropna: (7032, 21)


In [12]:
telco_data.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   str    
 1   gender            7032 non-null   str    
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   str    
 4   Dependents        7032 non-null   str    
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   str    
 7   MultipleLines     7032 non-null   str    
 8   InternetService   7032 non-null   str    
 9   OnlineSecurity    7032 non-null   str    
 10  OnlineBackup      7032 non-null   str    
 11  DeviceProtection  7032 non-null   str    
 12  TechSupport       7032 non-null   str    
 13  StreamingTV       7032 non-null   str    
 14  StreamingMovies   7032 non-null   str    
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  PaymentMeth

In [13]:
telco_data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [14]:
# Verify max tenure = 72
print("Max tenure:", telco_data.tenure.max())

Max tenure: 72


### Create Tenure Bins

The ML notebook uses a slightly different binning approach than the EDA notebook:
- Explicit bin edges: [0, 12, 24, 36, 48, 60, 72]
- `include_lowest=True` ensures that tenure=0 is captured in the first bin

In [15]:
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ['1-12', '13-24', '25-36', '37-48', '49-60', '61-72']

telco_data['tenure_bin'] = pd.cut(telco_data['tenure'], bins=bins, labels=labels, include_lowest=True)

print(telco_data[['tenure', 'tenure_bin']].head(10))
print(telco_data['tenure_bin'].value_counts().sort_index())

   tenure tenure_bin
0       1       1-12
1      34      25-36
2       2       1-12
3      45      37-48
4       2       1-12
5       8       1-12
6      22      13-24
7      10       1-12
8      28      25-36
9      62      61-72
tenure_bin
1-12     2175
13-24    1024
25-36     832
37-48     762
49-60     832
61-72    1407
Name: count, dtype: int64


In [16]:
telco_data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


## 5. Define Features and Target

We drop:
- `customerID`: non-predictive unique identifier
- `Churn`: target variable
- `tenure`: replaced by `tenure_bin`

Then apply one-hot encoding with `drop_first=True`.

In [17]:
y = telco_data['Churn']
X = telco_data.drop(columns=['customerID', 'Churn', 'tenure'])

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (7032, 19)
Shape of y: (7032,)


In [18]:
X

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,tenure_bin
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,1-12
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,25-36
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1-12
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,37-48
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,13-24
7039,Female,0,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,61-72
7040,Female,0,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,1-12
7041,Male,1,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1-12


In [19]:
# One-hot encode all categorical columns
X = pd.get_dummies(X, drop_first=True)

# Map target to binary
y = telco_data['Churn'].map({'No': 0, 'Yes': 1})

print("X shape after encoding:", X.shape)
print("Columns:", list(X.columns))

X shape after encoding: (7032, 34)
Columns: ['SeniorCitizen', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'tenure_bin_13-24', 'tenure_bin_25-36', 'tenure_bin_37-48', 'tenure_bin_49-60', 'tenure_bin_61-72']


In [20]:
X

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bin_13-24,tenure_bin_25-36,tenure_bin_37-48,tenure_bin_49-60,tenure_bin_61-72
0,0,29.85,29.85,False,True,False,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,...,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,...,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,...,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,True,True,True,True,False,True,False,...,False,True,False,False,True,True,False,False,False,False
7039,0,103.20,7362.90,False,True,True,True,False,True,True,...,False,True,True,False,False,False,False,False,False,True
7040,0,29.60,346.45,False,True,True,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
7041,1,74.40,306.60,True,True,False,True,False,True,True,...,False,True,False,False,True,False,False,False,False,False


In [21]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (5625, 34) X_test: (1407, 34)


## 6. Feature Scaling

**Why scale?**
- Many algorithms (logistic regression, SVM, gradient descent-based) are sensitive to feature magnitude
- Tree-based methods (DT, RF, XGBoost) are not sensitive to scale but it does not hurt
- Always fit the scaler on training data ONLY, then transform both train and test
  (to avoid data leakage from test set statistics entering the model)

**StandardScaler:** centers to mean=0, std=1
**MinMaxScaler:** scales to [0, 1] range

In [22]:
# StandardScaler
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)  # fit on train, transform train
X_test_sc = sc.transform(X_test)        # transform test using train statistics only

In [23]:
# Decision Tree with StandardScaler
model_dt2 = DecisionTreeClassifier()
model_dt2.fit(X_train_sc, y_train)
y_pred_dt2 = model_dt2.predict(X_test_sc)

print("=== Decision Tree + StandardScaler ===")
print(classification_report(y_test, y_pred_dt2))

=== Decision Tree + StandardScaler ===
              precision    recall  f1-score   support

           0       0.82      0.81      0.81      1033
           1       0.49      0.50      0.50       374

    accuracy                           0.73      1407
   macro avg       0.65      0.66      0.66      1407
weighted avg       0.73      0.73      0.73      1407



In [24]:
# MinMaxScaler (applied on top of StandardScaler output)
mms = MinMaxScaler()
X_train_mms = mms.fit_transform(X_train_sc)
X_test_mms = mms.transform(X_test_sc)

model_dt3 = DecisionTreeClassifier()
model_dt3.fit(X_train_mms, y_train)
y_pred_dt3 = model_dt3.predict(X_test_mms)

print("Decision Tree + MinMaxScaler")
print(classification_report(y_test, y_pred_dt3))

Decision Tree + MinMaxScaler
              precision    recall  f1-score   support

           0       0.82      0.80      0.81      1033
           1       0.49      0.52      0.50       374

    accuracy                           0.73      1407
   macro avg       0.65      0.66      0.66      1407
weighted avg       0.73      0.73      0.73      1407



## 7. Handling Class Imbalance with SMOTEENN

**SMOTEENN** combines two techniques:
- **SMOTE (Synthetic Minority Over-sampling Technique):** generates synthetic samples for the minority class
- **ENN (Edited Nearest Neighbours):** removes noisy samples near class boundaries

Result: upsamples minority class and removes borderline samples from majority class.

**Important:** Apply resampling ONLY to training data, never to test data.
Test data must represent the true distribution.

In [25]:
from imblearn.combine import SMOTEENN

sm = SMOTEENN()
# Resample the scaled training data
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_sc, y_train)

from collections import Counter
print("After SMOTEENN:", Counter(y_train_resampled))
# Classes are now more balanced

After SMOTEENN: Counter({1: 2938, 0: 2236})


In [26]:
# Decision Tree trained on SMOTEENN-resampled data
model_dt_smoteenn = DecisionTreeClassifier()
model_dt_smoteenn.fit(X_train_resampled, y_train_resampled)
y_pred_dt_smoteenn = model_dt_smoteenn.predict(X_test_sc)

print("=== Decision Tree + SMOTEENN ===")
print(classification_report(y_test, y_pred_dt_smoteenn))

=== Decision Tree + SMOTEENN ===
              precision    recall  f1-score   support

           0       0.89      0.67      0.77      1033
           1       0.46      0.78      0.58       374

    accuracy                           0.70      1407
   macro avg       0.68      0.73      0.67      1407
weighted avg       0.78      0.70      0.72      1407



In [27]:
# Random Forest with 500 trees + SMOTEENN
# n_estimators=500: more trees = more stable predictions but slower training
model_rf_smoteenn = RandomForestClassifier(n_estimators=500)
model_rf_smoteenn.fit(X_train_resampled, y_train_resampled)
y_pred_rf_smoteenn = model_rf_smoteenn.predict(X_test_sc)

print("Random Forest (500 trees) + SMOTEENN")
print(classification_report(y_test, y_pred_rf_smoteenn))

Random Forest (500 trees) + SMOTEENN
              precision    recall  f1-score   support

           0       0.91      0.71      0.79      1033
           1       0.50      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.70      1407
weighted avg       0.80      0.73      0.75      1407



## 8. XGBoost Models

XGBoost is a highly optimised gradient boosting implementation.
It builds trees sequentially where each tree corrects errors of the previous one.

In [29]:
from xgboost import XGBClassifier

# XGBoost without SMOTEENN
model_xgb = XGBClassifier(random_state=42)
model_xgb.fit(X_train_sc, y_train)
y_pred_xgb = model_xgb.predict(X_test_sc)

print("XGBoost (no resampling)")
print(classification_report(y_test, y_pred_xgb))

XGBoost (no resampling)
              precision    recall  f1-score   support

           0       0.83      0.88      0.85      1033
           1       0.60      0.51      0.55       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407



In [30]:
# XGBoost with SMOTEENN resampled data
model_xgb_smoteenn = XGBClassifier(random_state=42)
model_xgb_smoteenn.fit(X_train_resampled, y_train_resampled)
y_pred_xgb_smoteenn = model_xgb_smoteenn.predict(X_test_sc)

print("XGBoost + SMOTEENN")
print(classification_report(y_test, y_pred_xgb_smoteenn))

XGBoost + SMOTEENN
              precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407



### XGBoost with SMOTE (Oversampling only)

SMOTE generates synthetic minority class samples by interpolating between existing samples.
Unlike SMOTEENN, it does not remove any majority samples.

In [31]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", Counter(y_train))

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_sc, y_train)

print("After SMOTE:", Counter(y_train_smote))

Before SMOTE: Counter({0: 4130, 1: 1495})
After SMOTE: Counter({1: 4130, 0: 4130})


In [32]:
model_xgb_smote = XGBClassifier(random_state=42)
model_xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_xgb_smote = model_xgb_smote.predict(X_test_sc)

print("GBoost + SMOTE")
print(classification_report(y_test, y_pred_xgb_smote))

GBoost + SMOTE
              precision    recall  f1-score   support

           0       0.84      0.86      0.85      1033
           1       0.58      0.55      0.56       374

    accuracy                           0.77      1407
   macro avg       0.71      0.70      0.71      1407
weighted avg       0.77      0.77      0.77      1407



### XGBoost with ADASYN

ADASYN (Adaptive Synthetic Sampling) is similar to SMOTE but generates more
synthetic samples in regions that are harder to learn (near decision boundaries).

In [33]:
from imblearn.over_sampling import ADASYN

print("Before ADASYN:", Counter(y_train))

adasyn = ADASYN(random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train_sc, y_train)

print("After ADASYN:", Counter(y_train_adasyn))

Before ADASYN: Counter({0: 4130, 1: 1495})
After ADASYN: Counter({0: 4130, 1: 4096})


In [34]:
model_xgb_adasyn = XGBClassifier(random_state=42)
model_xgb_adasyn.fit(X_train_adasyn, y_train_adasyn)
y_pred_xgb_adasyn = model_xgb_adasyn.predict(X_test_sc)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb_adasyn))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb_adasyn))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb_adasyn))

Accuracy: 0.7746979388770433

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.85      0.85      1033
           1       0.58      0.57      0.57       374

    accuracy                           0.77      1407
   macro avg       0.71      0.71      0.71      1407
weighted avg       0.77      0.77      0.77      1407


Confusion Matrix:
 [[876 157]
 [160 214]]


### XGBoost with scale_pos_weight

`scale_pos_weight` is XGBoost's built-in way to handle imbalance.
Setting it to `negative_count / positive_count` tells XGBoost to weight
the minority class proportionally during training.
This avoids resampling entirely.

In [36]:
# Compute the weight ratio
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

scale_pos_weight: 2.76



This means minority class samples are weighted ~2.76x more heavily

In [37]:
model_xgb_weighted = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
model_xgb_weighted.fit(X_train_sc, y_train)
y_pred_weighted = model_xgb_weighted.predict(X_test_sc)

print("Accuracy:", accuracy_score(y_test, y_pred_weighted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_weighted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_weighted))

Accuracy: 0.7469793887704336

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.78      0.82      1033
           1       0.52      0.65      0.58       374

    accuracy                           0.75      1407
   macro avg       0.69      0.72      0.70      1407
weighted avg       0.77      0.75      0.76      1407


Confusion Matrix:
 [[807 226]
 [130 244]]


## 9. AdaBoost with Sample Weights

AdaBoost (Adaptive Boosting) builds weak classifiers sequentially,
giving more weight to misclassified samples in each round.

We pass `sample_weight` explicitly to tell AdaBoost to treat the
minority class (churners) with higher importance from the start.

**This is the model that gets saved as the final .pkl file.**

In [38]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
weight_positive = negative_count / positive_count if positive_count > 0 else 1

print(f"Negative class count: {negative_count}")
print(f"Positive class count: {positive_count}")
print(f"Weight for positive class: {weight_positive:.2f}")

Negative class count: 4130
Positive class count: 1495
Weight for positive class: 2.76


In [39]:
# Create sample weights array: minority class gets higher weight
sample_weight = np.where(y_train == 1, weight_positive, 1.0)

model_ada_weighted = AdaBoostClassifier(
    n_estimators=50,  # 50 weak learners
    random_state=42
)
model_ada_weighted.fit(X_train_sc, y_train, sample_weight=sample_weight)

y_pred_ada_weighted = model_ada_weighted.predict(X_test_sc)

print("AdaBoost with sample weighting - Accuracy:", accuracy_score(y_test, y_pred_ada_weighted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_ada_weighted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_ada_weighted))

AdaBoost with sample weighting - Accuracy: 0.7221037668798863

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.79      0.60       374

    accuracy                           0.72      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.72      0.74      1407


Confusion Matrix:
 [[720 313]
 [ 78 296]]


## 10. Hyperparameter Optimization with RandomizedSearchCV

**RandomizedSearchCV** samples a fixed number of hyperparameter combinations
from a defined distribution and selects the best using cross-validation.

It is faster than GridSearchCV (which tries every combination) while
still exploring the hyperparameter space broadly.

We optimize for F1 score instead of accuracy because the dataset is imbalanced.

In [40]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 400],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]
}

xgb = XGBClassifier(random_state=42, eval_metric='logloss')

search = RandomizedSearchCV(
    xgb,
    param_grid,
    n_iter=50,       # Try 50 random combinations
    cv=5,            # 5-fold cross-validation
    scoring='f1',    # Optimise for F1 score on minority class
    random_state=42,
    n_jobs=-1        # Use all CPU cores
)

search.fit(X_train_sc, y_train)

print("Best parameters:", search.best_params_)
print("Best CV F1 score:", search.best_score_)

Best parameters: {'subsample': 0.7, 'scale_pos_weight': np.float64(2.762541806020067), 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.8}
Best CV F1 score: 0.6381715173288723


In [41]:
best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test_sc)
print("\nTest Classification Report:\n", classification_report(y_test, y_pred_best))


Test Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.72      0.80      1033
           1       0.50      0.77      0.61       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.74      0.75      1407



In [42]:
# Save the best XGBoost model
xgb_save_path = os.path.join(MODELS_DIR, 'best_xgboost_churn_model.pkl')
joblib.dump(best_model, xgb_save_path)
print(f"Best XGBoost model saved to: {xgb_save_path}")

Best XGBoost model saved to: c:\projects\Customer_Churn_Prediction\models\best_xgboost_churn_model.pkl


## 11. Additional Models for Comparison

In [44]:
# Random Forest with class_weight='balanced'
# 'balanced' automatically adjusts weights inversely proportional to class frequencies
model_rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=300,
    max_depth=6
)
model_rf.fit(X_train_sc, y_train)
y_pred_rf = model_rf.predict(X_test_sc)

print("Random Forest (balanced, depth=6, 300 trees)")
print(classification_report(y_test, y_pred_rf))

Random Forest (balanced, depth=6, 300 trees)
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.72      0.74      1407



In [45]:
from catboost import CatBoostClassifier

# CatBoost handles categorical features natively and has built-in class balancing
model_cat = CatBoostClassifier(
    auto_class_weights='Balanced',
    verbose=0,
    random_state=42
)
model_cat.fit(X_train_sc, y_train)
y_pred_cat = model_cat.predict(X_test_sc)

print("CatBoost (auto balanced weights)")
print(classification_report(y_test, y_pred_cat))

CatBoost (auto balanced weights)
              precision    recall  f1-score   support

           0       0.88      0.74      0.81      1033
           1       0.50      0.73      0.60       374

    accuracy                           0.74      1407
   macro avg       0.69      0.73      0.70      1407
weighted avg       0.78      0.74      0.75      1407



In [47]:
from lightgbm import LGBMClassifier

# LightGBM: fast gradient boosting, leaf-wise tree growth
model_lgb = LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
model_lgb.fit(X_train_sc, y_train)
y_pred_lgb = model_lgb.predict(X_test_sc)

print("LightGBM (scale_pos_weight)")
print(classification_report(y_test, y_pred_lgb))

[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001806 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
LightGBM (scale_pos_weight)
              precision    recall  f1-score   support

           0       0.89      0.74      0.81      1033
           1       0.51      0.74      0.60       374

    accuracy                           0.74      1407
   macro avg       0.70      0.74      0.70      1407
weighted avg       0.79      0.74      0.75      1407



c:\projects\Customer_Churn_Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 12. Optuna Hyperparameter Optimization

**Optuna** is a more advanced hyperparameter optimization framework that uses
Bayesian-style search (Tree-structured Parzen Estimator by default).

Unlike RandomizedSearchCV which randomly samples, Optuna learns from
previous trials to explore more promising regions of the hyperparameter space.

This generally finds better parameters in fewer trials.

In [48]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

# Suppress Optuna verbose output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """
    Optuna objective function.
    Each trial proposes a set of hyperparameters, trains the model
    with 5-fold CV, and returns the mean F1 score.
    Optuna maximises this score across trials.
    """
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'scale_pos_weight': trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    model = XGBClassifier(**params)
    f1_scorer = make_scorer(f1_score, average='binary', pos_label=1)
    score = cross_val_score(model, X_train_sc, y_train, cv=5, scoring=f1_scorer, n_jobs=-1).mean()
    return score


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)  # Use 50 trials; increase to 100 for production

print("Best parameters:", study.best_params)
print("Best CV F1 score:", study.best_value)

Best parameters: {'max_depth': 5, 'learning_rate': 0.01669863104023512, 'n_estimators': 215, 'subsample': 0.6888616682776707, 'colsample_bytree': 0.6962404991384393, 'min_child_weight': 7, 'gamma': 0.3811444292472272, 'reg_alpha': 0.36744477682040216, 'reg_lambda': 0.9047076779319192, 'scale_pos_weight': np.float64(2.762541806020067)}
Best CV F1 score: 0.6406799913281955


In [49]:
# Train final model with Optuna's best hyperparameters
best_params_optuna = study.best_params.copy()
best_params_optuna['random_state'] = 42
best_params_optuna['eval_metric'] = 'logloss'

best_model_optuna = XGBClassifier(**best_params_optuna)
best_model_optuna.fit(X_train_sc, y_train)
y_pred_optuna = best_model_optuna.predict(X_test_sc)

print("Optuna-tuned XGBoost")
print(classification_report(y_test, y_pred_optuna))

Optuna-tuned XGBoost
              precision    recall  f1-score   support

           0       0.90      0.71      0.79      1033
           1       0.49      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.73      0.74      1407



In [50]:
# Save Optuna model
optuna_save_path = os.path.join(MODELS_DIR, 'best_optuna_churn_model.pkl')
joblib.dump(best_model_optuna, optuna_save_path)
print(f"Optuna model saved to: {optuna_save_path}")

Optuna model saved to: c:\projects\Customer_Churn_Prediction\models\best_optuna_churn_model.pkl


## 13. Save AdaBoost Model (Final Production Model)

The AdaBoost model with sample weighting is saved as the final `.pkl` file
because it is the model loaded by the Streamlit app.

In [51]:
ada_save_path = os.path.join(MODELS_DIR, 'ada_boost_churn_model.pkl')
joblib.dump(model_ada_weighted, ada_save_path)
print(f"AdaBoost model saved to: {ada_save_path}")

AdaBoost model saved to: c:\projects\Customer_Churn_Prediction\models\ada_boost_churn_model.pkl


## 14. Model Comparison Summary

| Model | Key Technique | Notes |
|---|---|---|
| Decision Tree (baseline) | No cleaning | Misleadingly high accuracy |
| Decision Tree + StandardScaler | Scaling | Minimal improvement for trees |
| Decision Tree + SMOTEENN | Resampling | Better recall for churners |
| Random Forest 500 + SMOTEENN | Ensemble | More stable than single tree |
| XGBoost | Gradient boosting | Strong baseline |
| XGBoost + SMOTEENN | Gradient boosting + resampling | Good churn recall |
| XGBoost + SMOTE | Oversampling only | Simpler alternative |
| XGBoost + ADASYN | Adaptive oversampling | Focus on hard samples |
| XGBoost scale_pos_weight | Built-in weighting | No resampling needed |
| AdaBoost + sample_weight | Boosting + weighting | **Saved as production model** |
| Random Forest (balanced) | Built-in weighting | Good interpretability |
| CatBoost | Auto balanced | Handles categoricals natively |
| LightGBM | Fast boosting | Efficient on larger datasets |
| XGBoost + RandomizedSearchCV | Hyperparameter tuning | Systematic search |
| XGBoost + Optuna | Bayesian tuning | Smarter search |

**Metric focus for imbalanced churn data:**
- Prioritize **Recall for class 1** (catching actual churners matters most in business)
- Use **F1 score** as the balance between precision and recall
- Accuracy alone is misleading when classes are imbalanced